# Einführung in das Prompt Engineering
Prompt Engineering ist der Prozess des Entwerfens und Optimierens von Prompts für Aufgaben der Verarbeitung natürlicher Sprache. Es umfasst die Auswahl der richtigen Prompts, die Anpassung ihrer Parameter und die Bewertung ihrer Leistung. Prompt Engineering ist entscheidend, um hohe Genauigkeit und Effizienz in NLP-Modellen zu erreichen. In diesem Abschnitt werden wir die Grundlagen des Prompt Engineerings anhand der OpenAI-Modelle zur Erkundung untersuchen.


### Übung 1: Tokenisierung
Erkunde die Tokenisierung mit tiktoken, einem Open-Source schnellen Tokenizer von OpenAI
Siehe [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) für weitere Beispiele.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### Übung 2: Überprüfen der Einrichtung des Microsoft Foundry Models-Schlüssels

> **Hinweis:** GitHub Models wird Ende Juli 2026 eingestellt. Diese Übung verwendet stattdessen [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), die denselben kostenlosen Testmodellkatalog und das Azure AI Inference SDK-Erlebnis bieten.

Führen Sie den folgenden Code aus, um zu überprüfen, ob Ihr Microsoft Foundry Models-Endpunkt korrekt eingerichtet ist. Der Code versucht lediglich eine einfache Basiseingabe und validiert die Ausgabe. Die Eingabe `oh say can you see` sollte ungefähr mit `by the dawn's early light..` abgeschlossen werden.


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

model_name = "gpt-4o-mini"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### Übung 3: Erfindungen
Untersuchen Sie, was passiert, wenn Sie das LLM bitten, Vervollständigungen für eine Eingabeaufforderung zu einem Thema zurückzugeben, das möglicherweise nicht existiert, oder zu Themen, die es möglicherweise nicht kennt, weil sie außerhalb seines vortrainierten Datensatzes liegen (neuere). Sehen Sie, wie sich die Antwort ändert, wenn Sie eine andere Eingabeaufforderung oder ein anderes Modell ausprobieren.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### Übung 4: Anweisungsbasiert 
Verwenden Sie die Variable "text", um den Hauptinhalt festzulegen 
und die Variable "prompt", um eine Anweisung in Bezug auf diesen Hauptinhalt zu geben.

Hier bitten wir das Modell, den Text für einen Zweitklässler zusammenzufassen


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### Übung 5: Komplexe Aufforderung 
Versuchen Sie eine Anfrage, die System-, Benutzer- und Assistenten-Nachrichten enthält 
Das System setzt den Assistenten-Kontext
Benutzer- & Assistenten-Nachrichten liefern Kontext für ein mehrstufiges Gespräch

Beachten Sie, wie die Assistenten-Persönlichkeit im Systemkontext auf „sarkastisch“ gesetzt ist. 
Versuchen Sie, einen anderen Persönlichkeitskontext zu verwenden. Oder probieren Sie eine andere Folge von Eingabe-/Ausgabemeldungen aus


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### Übung: Erkunde deine Intuition
Die obigen Beispiele geben dir Muster, die du verwenden kannst, um neue Eingabeaufforderungen zu erstellen (einfach, komplex, Anweisung usw.) – versuche, weitere Übungen zu erstellen, um einige der anderen Ideen zu erkunden, über die wir gesprochen haben, wie Beispiele, Hinweise und mehr.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
